<a href="https://colab.research.google.com/github/karsarobert/DeepLearning2026/blob/main/PTE_DL7_EXERCISES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning 2026 · Practice 7
## Inno-guided practice in Google Colab · 1.0 EN

Topics: housing-value regression, MNIST digit classification and autoencoder-based transaction anomaly detection.

**Use two interfaces:** edit and run code in Colab; discuss the work with your tutor in Innoagent.
Work on one cell at a time. Read its purpose, run or complete it, and send the requested evidence.
The tutor cannot automatically see your Colab notebook. Send a short relevant output rather than only “done”.

- **Prepared cell:** read the supplied code, run it and interpret the result.
- **Independent cell:** write the code yourself from the instructions. Ask for gradual help when needed.
- Correct errors before moving on. Do not start with “run all” while the independent cells are still empty.
- R_, M_ and A_ prefixes separate the three tasks.
- Plots appear below their Colab cells. No Innoagent file-tree refresh or whole-page browser reload is required.

This is a revised working copy of the original exercise notebook, with separate validation sets, corrected order and stable cell IDs.
The historical housing dataset is a limited teaching example, not a current property valuation tool.


## Using code examples

The original first code cell combined unrelated snippets. It is not a single executable program.
Open **DL_07_practice_EN.html** for detailed explanations and complete examples using different data.
These reminders assume that the illustrative arrays already exist:

```python
# General forms for selecting a column and a two-dimensional sample:
column = table[:, 2]
one_sample = table[4:5]

# Apply an already fitted scaler to new inputs:
new_scaled = scaler.transform(new_data)

# A hidden layer and dropout in a different example:
hidden = layers.Dense(8, activation="relu")
dropout = layers.Dropout(0.2)

# The index of the largest output value:
class_index = np.argmax(one_output_vector)
```

Choose shapes, target formats and output sizes for the actual task.


## [E01] Environment and working method

**Mode: prepared cell.**

This cell imports the libraries and sets the random seed. It does not train a model. Use Colab to edit and run code; use Innoagent to discuss it. R_, M_ and A_ prefixes keep the three tasks separate.

**Your task:** Run this cell in Colab. Send the version numbers and the final readiness message.

**Send back to Inno:** TensorFlow and NumPy versions, plus the E01 readiness message.


In [ ]:
# [E01]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pathlib import Path

keras.utils.set_random_seed(42)
results = {}
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("E01: the Colab environment is ready.")


## [R01] Load the housing data

**Mode: prepared cell.**

Each row in the historical Boston dataset describes a district using 13 features. The target is its median home value in thousands of historical US dollars. After loading the official NPZ file, we will create our own split. A manually uploaded NPZ file is only a download fallback.

**Your task:** Run the loader. Report the feature and target shapes, and explain what the target represents.

**Send back to Inno:** Both shapes and the meaning of the target in your own words.


In [ ]:
# [R01]
R_data_path = Path("boston_housing.npz")
if not R_data_path.exists():
    R_data_path = Path(keras.utils.get_file(
        "boston_housing.npz",
        origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/boston_housing.npz",
        file_hash="f553886a1f8d56431e820c5b82552d9d95cfcb96d1e678153f8839538947dff5"
    ))
with np.load(R_data_path, allow_pickle=False) as data:
    R_X = data["x"].astype("float32")
    R_y = data["y"].astype("float32")
print("R01 input:", R_X.shape, "target:", R_y.shape)
print("First three targets [thousand historical USD]:", R_y[:3])


## [R02] Inspect the data

**Mode: prepared cell.**

A DataFrame makes the features readable. Column minima and maxima reveal their different scales. isfinite checks for missing or infinite numeric values.

**Your task:** Run the cell. Identify two actual column ranges and explain why scaling will be useful.

**Send back to Inno:** Two column ranges from your output and a brief explanation.


In [ ]:
# [R02]
R_feature_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE",
           "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"]
R_table = pd.DataFrame(R_X, columns=R_feature_names)
print(R_table.head(3))
print(pd.DataFrame({"minimum": R_table.min(), "maximum": R_table.max()}))
print("All values are finite:", np.isfinite(R_X).all(), np.isfinite(R_y).all())


## [R03] Training, validation and test sets

**Mode: prepared cell.**

We first keep 60% for training and split the remainder in half. Validation guides early stopping; the final test is reserved for evaluation. Integer rounding makes validation and test counts slightly different.

**Your task:** Run the split. Send the three sample counts and identify the subset used for early stopping.

**Send back to Inno:** Three sample counts and the role of validation.


In [ ]:
# [R03]
R_X_train, R_X_remaining, R_y_train, R_y_remaining = train_test_split(
    R_X, R_y, test_size=0.4, random_state=42
)
R_X_val, R_X_test, R_y_val, R_y_test = train_test_split(
    R_X_remaining, R_y_remaining, test_size=0.5, random_state=42
)
print("R03 training / validation / test:",
      len(R_X_train), len(R_X_val), len(R_X_test))


## [R04] Scale without data leakage

**Mode: independent cell.**

A MinMaxScaler learns a minimum and maximum for each input column. Fit it only on training inputs. Keep target prices in their original unit.

**Your task:** Create R_scaler as a MinMaxScaler. Produce R_X_train_s, R_X_val_s and R_X_test_s. Use fit_transform only on training inputs and transform on the other two subsets. Print all three shapes and n_samples_seen_.

**Send back to Inno:** Your scaling code, three shapes and the number of fitting samples.


In [ ]:
# [R04]
# Independent coding: Scale without data leakage
# Write the code described above.
# Use the variable names specified in the instructions.


## [R05] Build the regression network

**Mode: independent cell.**

The network predicts one real value from 13 features. Hidden layers use ReLU. A linear output does not constrain the predicted price to the interval 0–1.

**Your task:** Build R_model as a Sequential model. Set its input size from the scaled training feature count, then add 64 and 32 ReLU units and one linear output. Display the model summary.

**Send back to Inno:** Model code, the final output shape and why one output is sufficient.


In [ ]:
# [R05]
# Independent coding: Build the regression network
# Write the code described above.
# Use the variable names specified in the instructions.


## [R06] Loss and early stopping

**Mode: independent cell.**

compile configures learning; it does not train. MAE averages absolute errors. Early stopping selects an appropriate training state using validation loss.

**Your task:** Compile R_model with adam and mae. Create R_stop using EarlyStopping: monitor val_loss, min_delta=0.001, patience=12, and restore the best weights. Print the loss and monitor names.

**Send back to Inno:** Compile and callback code, plus what 12 epochs without improvement means.


In [ ]:
# [R06]
# Independent coding: Loss and early stopping
# Write the code described above.
# Use the variable names specified in the instructions.


## [R07] Train the regression model

**Mode: prepared cell.**

fit updates weights using training features and their target values. Validation does not directly update weights. The 150-epoch budget is an upper limit.

**Your task:** Run training and wait for completion. Report the completed epoch count and the lowest validation loss.

**Send back to Inno:** The two final summary values after training finishes.


In [ ]:
# [R07]
R_history = R_model.fit(
    R_X_train_s, R_y_train,
    validation_data=(R_X_val_s, R_y_val),
    epochs=150, batch_size=32, callbacks=[R_stop], verbose=1
)
print("R07 completed epochs:", len(R_history.history["loss"]))
print("Lowest validation MAE:", min(R_history.history["val_loss"]))


## [R08] Read the learning curves

**Mode: prepared cell.**

The plot shows training and validation MAE. Joint improvement and a persistent separation suggest different learning situations. Focus on directions and relationships rather than an exact epoch number.

**Your task:** Run the plot. Describe one specific relationship between the two curves.

**Send back to Inno:** A screenshot or a specific description of the actual curves.


In [ ]:
# [R08]
plt.figure(figsize=(8, 4))
plt.plot(R_history.history["loss"], label="Training MAE")
plt.plot(R_history.history["val_loss"], label="Validation MAE")
plt.xlabel("Epoch index (starting at 0)")
plt.ylabel("MAE [thousand historical USD]")
plt.legend()
plt.tight_layout()
plt.show()


## [R09] Final test and a baseline

**Mode: prepared cell.**

We now evaluate the reserved test set. Compare the network with a constant prediction equal to the median training target, a natural MAE baseline.

**Your task:** Run the cell. Report both MAEs and explain the network error in the correct unit.

**Send back to Inno:** Network MAE, baseline MAE and an interpretation of the network value.


In [ ]:
# [R09]
R_mae = float(R_model.evaluate(R_X_test_s, R_y_test, verbose=0))
R_pred = R_model.predict(R_X_test_s, verbose=0).ravel()
R_baseline = np.full_like(R_y_test, np.median(R_y_train))
R_baseline_mae = mean_absolute_error(R_y_test, R_baseline)
print(f"R09 network MAE: {R_mae:.3f} thousand historical USD")
print(f"Baseline MAE: {R_baseline_mae:.3f} thousand historical USD")
print(pd.DataFrame({"actual": R_y_test[:5], "predicted": R_pred[:5]}))
results["regression"] = {"mae": R_mae, "baseline_mae": float(R_baseline_mae)}


## [R10] A new input and the same scaler

**Mode: independent cell.**

Use a raw test sample to illustrate the path of a new input. This is not an additional independent test. Preserve the sample-count dimension even for one row.

**Your task:** Select the first raw R_X_test row as a two-dimensional slice named R_new. Transform it with the fitted R_scaler to create R_new_s, then predict with R_model. Print input and output shapes and the prediction.

**Send back to Inno:** Your code, both shapes and why transform is used instead of fit_transform.


In [ ]:
# [R10]
# Independent coding: A new input and the same scaler
# Write the code described above.
# Use the variable names specified in the instructions.


## [M01] MNIST and a smaller training subset

**Mode: prepared cell.**

MNIST contains 28×28 images of handwritten digits. To keep the exercise manageable, choose a stratified subset of 12000 from the 60000 original training images: 10000 for training and 2000 for validation. Keep the original 10000 test images separate.

**Your task:** Run the loader. Report the three image-array shapes and the range of integer labels.

**Send back to Inno:** Three shapes and the available labels.


In [ ]:
# [M01]
if Path("mnist.npz").exists():
    with np.load("mnist.npz", allow_pickle=False) as data:
        M_all_images = data["x_train"]
        M_all_labels = data["y_train"]
        M_test_images = data["x_test"]
        M_y_test = data["y_test"]
else:
    (M_all_images, M_all_labels), (M_test_images, M_y_test) = (
        keras.datasets.mnist.load_data()
    )
M_subset_images, _, M_subset_labels, _ = train_test_split(
    M_all_images, M_all_labels, train_size=12000,
    random_state=42, stratify=M_all_labels
)
M_train_images, M_val_images, M_y_train, M_y_val = train_test_split(
    M_subset_images, M_subset_labels, test_size=2000,
    random_state=42, stratify=M_subset_labels
)
print("M01 training:", M_train_images.shape)
print("Validation:", M_val_images.shape, "test:", M_test_images.shape)
print("Labels:", np.unique(M_y_train))


## [M02] One image and its true label

**Mode: prepared cell.**

Index 3 selects the fourth image. M_y_test still stores integer labels. The label records which digit the dataset says the image represents.

**Your task:** Run the cell. State the ordinal number of the image and its true label.

**Send back to Inno:** The image position and the integer label from your output.


In [ ]:
# [M02]
M_index = 3  # The fourth test image, using zero-based indexing.
plt.imshow(M_test_images[M_index], cmap="gray", vmin=0, vmax=255)
plt.title(f"True label: {M_y_test[M_index]}")
plt.axis("off")
plt.show()
print("M02 index:", M_index, "label:", int(M_y_test[M_index]))


## [M03] Normalize and flatten

**Mode: independent cell.**

Raw uint8 pixels range from 0 to 255. Converting to float32 and dividing by 255 gives values in 0–1. A 28×28 image retains all 784 values when reshaped into a vector.

**Your task:** From the raw M_train_images, M_val_images and M_test_images, create float32 arrays M_X_train, M_X_val and M_X_test. Divide by 255 and reshape each image to 784 values. Print the three shapes and the training minimum and maximum.

**Send back to Inno:** Your code, the shapes, value range and an explanation of -1.


In [ ]:
# [M03]
# Independent coding: Normalize and flatten
# Write the code described above.
# Use the variable names specified in the instructions.


## [M04] One-hot target vectors

**Mode: independent cell.**

A one-hot vector has one position per class. The correct class position contains 1 and other positions contain 0. Preserve the original integer label arrays.

**Your task:** Use keras.utils.to_categorical to create M_y_train_oh, M_y_val_oh and M_y_test_oh with ten classes. Print the training target shape and the first integer label with its one-hot vector.

**Send back to Inno:** Your code, the (10000, 10) shape and one matching integer/vector pair.


In [ ]:
# [M04]
# Independent coding: One-hot target vectors
# Write the code described above.
# Use the variable names specified in the instructions.


## [M05] Build the digit classifier

**Mode: independent cell.**

Two dense hidden layers process 784 pixels. Dropout randomly zeros some activations during training. Ten softmax outputs correspond to the ten digits.

**Your task:** Create M_model with 784 inputs, 128 ReLU units, dropout 0.3, 64 ReLU units, another dropout 0.3, and ten softmax outputs. Display its summary.

**Send back to Inno:** Model code, the final shape and the reason for ten outputs.


In [ ]:
# [M05]
# Independent coding: Build the digit classifier
# Write the code described above.
# Use the variable names specified in the instructions.


## [M06] Choose a classification loss

**Mode: independent cell.**

Our labels are one-hot vectors and the model uses softmax, so categorical_crossentropy is appropriate. Accuracy measures correct class decisions and is different from loss.

**Your task:** Compile M_model with adam, a loss suitable for one-hot targets and accuracy. Create M_stop monitoring val_loss with patience=4 and restore_best_weights=True. Print the loss name.

**Send back to Inno:** Compile and callback code with a reason for the chosen loss.


In [ ]:
# [M06]
# Independent coding: Choose a classification loss
# Write the code described above.
# Use the variable names specified in the instructions.


## [M07] Train the classifier

**Mode: prepared cell.**

Use at most 20 epochs and a batch size of 256. Validation images and targets are separate. verbose=1 shows progress; the final test is still unused.

**Your task:** Run training and wait for it to finish. Send the epoch count and the highest validation accuracy.

**Send back to Inno:** The completed epoch count and validation accuracy from your run.


In [ ]:
# [M07]
M_history = M_model.fit(
    M_X_train, M_y_train_oh,
    validation_data=(M_X_val, M_y_val_oh),
    epochs=20, batch_size=256, callbacks=[M_stop], verbose=1
)
print("M07 epochs:", len(M_history.history["loss"]))
print("Highest validation accuracy:", max(M_history.history["val_accuracy"]))


## [M08] Plot loss and accuracy separately

**Mode: prepared cell.**

The left plot shows cross-entropy and the right plot shows accuracy. Dropout is active during training and inactive during validation, so the relative positions alone do not prove an error.

**Your task:** Run the plots. Make one specific training-versus-validation observation on either plot.

**Send back to Inno:** An image or a specific description based on your plots.


In [ ]:
# [M08]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(M_history.history["loss"], label="Training")
ax[0].plot(M_history.history["val_loss"], label="Validation")
ax[0].set_title("Cross-entropy")
ax[1].plot(M_history.history["accuracy"], label="Training")
ax[1].plot(M_history.history["val_accuracy"], label="Validation")
ax[1].set_title("Accuracy")
for axis_obj in ax:
    axis_obj.set_xlabel("Epoch index (starting at 0)")
    axis_obj.legend()
plt.tight_layout()
plt.show()


## [M09] Test performance and one digit prediction

**Mode: independent cell.**

predict returns ten values for one image. The position of the largest value is the predicted digit. Read the true integer label from the preserved M_y_test array.

**Your task:** Evaluate M_model on M_X_test and M_y_test_oh, storing M_loss and M_acc. Pass the image at M_index to predict using a two-dimensional slice, then use np.argmax to store M_predicted_digit. Print accuracy, the ten outputs, the predicted digit and its true label.

**Send back to Inno:** Your code, accuracy, output vector, predicted digit and true digit.


In [ ]:
# [M09]
# Independent coding: Test performance and one digit prediction
# Write the code described above.
# Use the variable names specified in the instructions.


## [M10] Inspect classification errors

**Mode: prepared cell.**

The complete test output is 10000×10. argmax(axis=1) chooses a class for each image. Confusion-matrix rows are true labels and columns are predictions. The code displays a wrong prediction if one exists.

**Your task:** Run the cell. Interpret one matrix cell using its true label, predicted label and count.

**Send back to Inno:** One actual matrix cell or a misclassified image label pair, plus the full test error count.


In [ ]:
# [M10]
M_all_probabilities = M_model.predict(M_X_test, verbose=0)
M_pred = np.argmax(M_all_probabilities, axis=1)
M_cm = confusion_matrix(M_y_test, M_pred, labels=np.arange(10))
ConfusionMatrixDisplay(M_cm, display_labels=np.arange(10)).plot(cmap="Blues")
plt.title("True label: row; predicted label: column")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()
M_wrong_indices = np.flatnonzero(M_pred != M_y_test)
print("M10 number of errors:", len(M_wrong_indices))
if len(M_wrong_indices):
    i = int(M_wrong_indices[0])
    plt.imshow(M_test_images[i], cmap="gray")
    plt.title(f"True: {M_y_test[i]}, predicted: {M_pred[i]}")
    plt.axis("off")
    plt.show()
results["mnist"] = {"accuracy": float(M_acc), "errors": int(len(M_wrong_indices))}


## [A01] Synthetic transactions and labels

**Mode: prepared cell.**

As in the source notebook, generate 1000 normal and 50 shifted anomalous samples with five features. These are artificial numbers rather than real transaction fields. Label 0 means normal and 1 means anomalous; this simple dataset does not represent real bank fraud complexity.

**Your task:** Run the cell. Report both class counts, the input shape and the anomaly label.

**Send back to Inno:** Class meanings, counts and input shape.


In [ ]:
# [A01]
A_rng = np.random.default_rng(42)
A_normal = A_rng.random((1000, 5)) * 10
A_anomalies = A_rng.random((50, 5)) * 10 + 5
A_X = np.vstack([A_normal, A_anomalies]).astype("float32")
A_y = np.concatenate([np.zeros(1000), np.ones(50)]).astype("int32")
A_label_values, A_label_counts = np.unique(A_y, return_counts=True)
print("A01 labels and counts:", dict(zip(A_label_values, A_label_counts)))
print("Input:", A_X.shape)


## [A02] Split before fitting a scaler

**Mode: prepared cell.**

Stratification preserves class proportions. These are still raw inputs: no scaler has been fitted to the full dataset. Select normal samples after splitting.

**Your task:** Run the split. Send the three subset sizes and the class counts in the test set.

**Send back to Inno:** 630 / 210 / 210 and the test label counts.


In [ ]:
# [A02]
A_X_train, A_X_remaining, A_y_train, A_y_remaining = train_test_split(
    A_X, A_y, test_size=0.4, random_state=42, stratify=A_y
)
A_X_val, A_X_test, A_y_val, A_y_test = train_test_split(
    A_X_remaining, A_y_remaining, test_size=0.5,
    random_state=42, stratify=A_y_remaining
)
print("A02 sample counts:", len(A_X_train), len(A_X_val), len(A_X_test))
print("Test label distribution:", np.unique(A_y_test, return_counts=True))


## [A03] Normal training data and separate calibration

**Mode: independent cell.**

Select normal rows using label 0. Split normal validation samples into one subset for early stopping and another for threshold calibration, so the threshold uses neither final test data nor the early-stopping subset.

**Your task:** Create A_normal_train and A_normal_val from normal training and validation rows. Split A_normal_val equally into A_normal_stop and A_normal_cal using train_test_split with random_state=42. Print the three resulting shapes.

**Send back to Inno:** Your filtering code and the three shapes.


In [ ]:
# [A03]
# Independent coding: Normal training data and separate calibration
# Write the code described above.
# Use the variable names specified in the instructions.


## [A04] Learn the normal input scale

**Mode: independent cell.**

Apply the same scaler to normal stopping, normal calibration and mixed test inputs. Scaled anomalies may exceed 1. Do not clip them: the deviation can carry information.

**Your task:** Create A_scaler as a MinMaxScaler, then A_train_s, A_stop_s, A_cal_s and A_test_s. Fit only on A_normal_train. Print n_samples_seen_ and all four shapes.

**Send back to Inno:** Your code, the 600-sample fitting count and four shapes.


In [ ]:
# [A04]
# Independent coding: Learn the normal input scale
# Write the code described above.
# Use the variable names specified in the instructions.


## [A05] Build an autoencoder with a bottleneck

**Mode: independent cell.**

Reconstruct five features through a three-value middle layer. These artificial normal inputs are not intrinsically low-dimensional, so compression can also cause normal reconstruction error. Five sigmoid outputs match the normal training range 0–1 and cannot exactly reproduce inputs outside that range.

**Your task:** Build A_model: 5 inputs → 16 ReLU → 3 linear → 16 ReLU → 5 sigmoid outputs. Compile with adam and mse, and show its summary.

**Send back to Inno:** Model code and the meaning of the five outputs and three-value middle layer.


In [ ]:
# [A05]
# Independent coding: Build an autoencoder with a bottleneck
# Write the code described above.
# Use the variable names specified in the instructions.


## [A06] Train inputs to reconstruct themselves

**Mode: prepared cell.**

Normal training inputs also serve as targets. The stopping subset contains normal samples only. Calibration and test data are not passed to fit.

**Your task:** Run training. Report the epoch count and explain why the first two fit arguments are the same array.

**Send back to Inno:** Completed epoch count and an explanation of input=target.


In [ ]:
# [A06]
A_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
A_history = A_model.fit(
    A_train_s, A_train_s,
    validation_data=(A_stop_s, A_stop_s),
    epochs=100, batch_size=32, callbacks=[A_stop], verbose=1
)
print("A06 epochs:", len(A_history.history["loss"]))
print("Lowest normal validation MSE:", min(A_history.history["val_loss"]))


## [A07] One calibration error per sample

**Mode: independent cell.**

Reconstruct the separate normal calibration samples. Average the five squared feature errors along each row to obtain one error per sample. A single whole-array average cannot support per-sample decisions.

**Your task:** Use A_model.predict on A_cal_s to create A_cal_recon. Compute per-sample MSE as A_cal_errors with np.mean along the correct axis. Print both shapes and the first five errors.

**Send back to Inno:** Your code, reconstruction shape, error shape and axis explanation.


In [ ]:
# [A07]
# Independent coding: One calibration error per sample
# Write the code described above.
# Use the variable names specified in the instructions.


## [A08] Fix the threshold before testing

**Mode: independent cell.**

Choose the 97th percentile of normal calibration errors. This is a preselected teaching rule, not a guarantee of 3% false alarms on every future dataset. Mean plus two standard deviations is another possible heuristic, not a universal law.

**Your task:** Create A_threshold using np.quantile at 0.97 on A_cal_errors. Print the threshold and calibration count. Explain why test errors are not used.

**Send back to Inno:** Your code, threshold, 100 calibration samples and the independence argument.


In [ ]:
# [A08]
# Independent coding: Fix the threshold before testing
# Write the code described above.
# Use the variable names specified in the instructions.


## [A09] Mixed test decisions and metrics

**Mode: independent cell.**

The 210 test samples contain both classes. An error larger than the fixed threshold produces decision 1. Use true labels only to evaluate the resulting decisions.

**Your task:** Reconstruct A_test_s as A_test_recon, compute per-sample MSE as A_test_errors, and create integer 0/1 decisions A_pred using A_threshold. Print a classification_report with labels=[0, 1], target_names=["Normal", "Anomaly"] and zero_division=0.

**Send back to Inno:** Your code and the report, including precision and recall in the Anomaly row.


In [ ]:
# [A09]
# Independent coding: Mixed test decisions and metrics
# Write the code described above.
# Use the variable names specified in the instructions.


## [A10] Error distributions and mistakes

**Mode: prepared cell.**

The confusion matrix summarizes correct and incorrect decisions. Histograms show reconstruction errors for the two true classes; the vertical line is the previously fixed threshold. Do not adjust it after inspecting the test.

**Your task:** Run the cell. Send TN, FP, FN and TP, and interpret one false alarm or missed anomaly.

**Send back to Inno:** Four actual counts and their interpretation, with the threshold unchanged.


In [ ]:
# [A10]
A_cm = confusion_matrix(A_y_test, A_pred, labels=[0, 1])
A_tn, A_fp, A_fn, A_tp = A_cm.ravel()
print("A10 TN, FP, FN, TP:", A_tn, A_fp, A_fn, A_tp)
ConfusionMatrixDisplay(A_cm, display_labels=["Normal", "Anomaly"]).plot(cmap="Blues")
plt.title("True label: row; predicted label: column")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
A_bins = np.histogram_bin_edges(A_test_errors, bins=25)
plt.hist(A_test_errors[A_y_test == 0], bins=A_bins, alpha=0.65, label="Normal")
plt.hist(A_test_errors[A_y_test == 1], bins=A_bins, alpha=0.65, label="Anomaly")
plt.axvline(A_threshold, color="black", linestyle="--", label="Fixed threshold")
plt.xlabel("Reconstruction MSE per sample")
plt.ylabel("Number of samples")
plt.legend()
plt.tight_layout()
plt.show()

A_precision = float(A_tp / (A_tp + A_fp)) if A_tp + A_fp else 0.0
A_recall = float(A_tp / (A_tp + A_fn)) if A_tp + A_fn else 0.0
results["anomaly"] = {"precision": A_precision, "recall": A_recall,
                        "TN": int(A_tn), "FP": int(A_fp),
                        "FN": int(A_fn), "TP": int(A_tp)}


## [Z01] Save and conclude

**Mode: prepared cell.**

The three tasks use different targets and metrics. This summary collects results from your own runs. Save your completed notebook using Colab’s save or download options.

**Your task:** Run the summary, send it to Inno, and save your completed notebook.

**Send back to Inno:** All three result groups and confirmation that the notebook was saved.


In [ ]:
# [Z01]
print("Z01: results from your runs")
for task_name, metrics in results.items():
    print(task_name, metrics)
print("Save your completed notebook with its code and outputs.")


## Closing reflection

Inno will ask these questions one at a time:

1. How did the target differ between regression, classification and the autoencoder?
2. Why did we not choose scaling, a model or a threshold from the final test set?
3. What do precision and recall mean in your own anomaly results?

There is no further required task or homework. Record any unfinished part accurately with your tutor.
